# Lab 011 — Read a tree split on real credit data

**Lesson:** [`lessons/0011-decision-trees-partitions.html`](../lessons/0011-decision-trees-partitions.html) · **Phase / Year:** Year 1 · Q2

**Dataset tier:** A — OpenML German credit (`credit_g`) via `relkit`

**Skill you are practising:** fit a depth-1 decision tree on real data, read its partition rule, and verify the Gini impurity reduction for that split.

**Exit criteria:** EXIT TICKET prints split feature name, threshold, child class rates, and hand-computed ΔGini matching sklearn within tolerance.

---

- **PROVIDED** — run only
- **TODO** — fill blanks
- **CHECK** — do not edit


## Setup — PROVIDED


In [ ]:
# PROVIDED
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()  # labs/))

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder

from relkit.data import load_tier_a

RS = 0
print("relkit ok")


## Data — PROVIDED (Tier A)

Real German credit data from OpenML — mixed numeric + categorical columns, imbalanced default rate.


In [ ]:
# PROVIDED
X, y = load_tier_a("credit_g")
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

X_num = X[num_cols].copy()
for c in cat_cols:
    X_num[c] = LabelEncoder().fit_transform(X[c].astype(str))

feat_names = list(X_num.columns)
print(f"rows={len(y)}  pos_rate={y.mean():.3f}  features={len(feat_names)}")


## Task 1 — Fit depth-1 tree — TODO

Fit a `DecisionTreeClassifier(max_depth=1, random_state=RS)` on the encoded matrix.


In [ ]:
# TODO
tree = DecisionTreeClassifier(max_depth=____, random_state=RS)
tree.fit(____, y)

t = tree.tree_
split_feat_idx = t.feature[0]
split_thresh = t.threshold[0]
split_name = feat_names[split_feat_idx]
print(f"split: {split_name} <= {split_thresh:.4f}")


In [ ]:
# CHECK
assert tree.get_depth() == 1, "Use max_depth=1."
assert split_feat_idx >= 0, "Tree must pick a split feature."
print("Task 1 ok — depth-1 tree fitted.")


## Task 2 — Child predictions — TODO

Read the predicted class in each child (nodes 1=left, 2=right in sklearn's array layout).


In [ ]:
# TODO — majority class in left/right child (0 or 1)
left_mask = X_num.iloc[:, split_feat_idx] <= split_thresh
right_mask = ~left_mask

left_rate = y[left_mask].mean()
right_rate = y[right_mask].mean()
print(f"left  n={left_mask.sum():4d}  pos_rate={left_rate:.3f}")
print(f"right n={right_mask.sum():4d}  pos_rate={right_rate:.3f}")


In [ ]:
# CHECK
assert left_mask.sum() + right_mask.sum() == len(y)
print("Task 2 ok — partition covers all rows.")


## Task 3 — Hand-compute ΔGini — TODO (crucial fragment)

Implement Gini for a binary vector and compute impurity reduction for the split above.


In [ ]:
def gini(y_vec):
    """Gini impurity for binary labels in {0,1}."""
    if len(y_vec) == 0:
        return 0.0
    p = y_vec.mean()
    return ____

# TODO: parent and weighted child Gini
G_parent = gini(y)
G_left = gini(y[left_mask])
G_right = gini(y[right_mask])
w_left = left_mask.mean()
w_right = right_mask.mean()
delta_g = ____

print(f"G_parent={G_parent:.4f}  delta_gini={delta_g:.4f}")


In [ ]:
# CHECK
assert 0 <= G_parent <= 0.5, "Binary Gini should be in [0, 0.5]."
assert delta_g > 0, "The chosen split should reduce impurity."
sklearn_imp = tree.tree_.impurity[0] - (
    (left_mask.sum()/len(y))*tree.tree_.impurity[1]
    + (right_mask.sum()/len(y))*tree.tree_.impurity[2]
)
assert abs(delta_g - sklearn_imp) < 0.02, f"Hand ΔGini {delta_g:.4f} vs sklearn {sklearn_imp:.4f}"
print("Task 3 ok — hand ΔGini matches sklearn.")


## Exit ticket — TODO


In [ ]:
# TODO: complete takeaway
print("=== EXIT TICKET — Lesson 011 ===")
print(f"split feature : {split_name}")
print(f"threshold     : {split_thresh:.4f}")
print(f"left pos_rate : {left_rate:.3f}  right pos_rate: {right_rate:.3f}")
print(f"delta Gini    : {delta_g:.4f}")
print()
print("takeaway:", "____")
